In [1]:
# Cell 1 — Import libraries
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os

load_dotenv()
password = quote_plus(os.getenv('DB_PASSWORD'))
DB_URL = f"postgresql://{os.getenv('DB_USER')}:{password}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
engine = create_engine(DB_URL)

print("✅ Connected to database!")

✅ Connected to database!


In [2]:
# Cell 2 — Load all tables from database
customers = pd.read_sql("SELECT * FROM customers", engine)
orders = pd.read_sql("SELECT * FROM orders", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)
sellers = pd.read_sql("SELECT * FROM sellers", engine)
payments = pd.read_sql("SELECT * FROM order_payments", engine)
reviews = pd.read_sql("SELECT * FROM order_reviews", engine)

print("✅ All tables loaded!")
print(f"customers:  {len(customers):,} rows")
print(f"orders:     {len(orders):,} rows")
print(f"order_items:{len(order_items):,} rows")

✅ All tables loaded!
customers:  99,441 rows
orders:     99,441 rows
order_items:112,650 rows


In [3]:
# Cell 3 — Check for missing values in every table
tables = {
    'customers': customers,
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews
}

for name, df in tables.items():
    missing = df.isnull().sum().sum()
    print(f"{name:15} → {missing:,} missing values")

customers       → 0 missing values
orders          → 4,908 missing values
order_items     → 0 missing values
products        → 2,448 missing values
sellers         → 0 missing values
payments        → 0 missing values
reviews         → 145,903 missing values


In [4]:
# Where exactly are orders missing values?
print("=== ORDERS missing values ===")
print(orders.isnull().sum())
print(f"\nTotal rows: {len(orders):,}")

=== ORDERS missing values ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Total rows: 99,441


In [5]:
# Where exactly are products missing values?
print("=== PRODUCTS missing values ===")
print(products.isnull().sum())

=== PRODUCTS missing values ===
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [6]:
# Where exactly are reviews missing values?
print("=== REVIEWS missing values ===")
print(reviews.isnull().sum())

=== REVIEWS missing values ===
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [8]:
# Fix orders — using CORRECT column names from our data
orders['order_approved_at'] = orders['order_approved_at'].fillna('pending')
orders['order_delivered_carrier_date'] = orders['order_delivered_carrier_date'].fillna('not delivered')
orders['order_delivered_customer_date'] = orders['order_delivered_customer_date'].fillna('not delivered')

# Fix products
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)
products['product_weight_g'] = products['product_weight_g'].fillna(0)
products['product_length_cm'] = products['product_length_cm'].fillna(0)
products['product_height_cm'] = products['product_height_cm'].fillna(0)
products['product_width_cm'] = products['product_width_cm'].fillna(0)

# Fix reviews — comment columns are optional so empty string is fine
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')

print("✅ Missing values fixed!")

# Verify all missing values are gone
for name, df in tables.items():
    missing = df.isnull().sum().sum()
    print(f"{name:15} → {missing:,} missing values remaining")

✅ Missing values fixed!
customers       → 0 missing values remaining
orders          → 0 missing values remaining
order_items     → 0 missing values remaining
products        → 0 missing values remaining
sellers         → 0 missing values remaining
payments        → 0 missing values remaining
reviews         → 0 missing values remaining


In [9]:
# Convert date columns to proper datetime format
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("✅ Date columns converted!")
print(orders.dtypes)

✅ Date columns converted!
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [10]:
for name, df in tables.items():
    dupes = df.duplicated().sum()
    print(f"{name:15} → {dupes:,} duplicate rows")

customers       → 0 duplicate rows
orders          → 0 duplicate rows
order_items     → 0 duplicate rows
products        → 0 duplicate rows
sellers         → 0 duplicate rows
payments        → 0 duplicate rows
reviews         → 0 duplicate rows


In [11]:
orders.to_sql('orders', engine, if_exists='replace', index=False)
products.to_sql('products', engine, if_exists='replace', index=False)
reviews.to_sql('order_reviews', engine, if_exists='replace', index=False)

print("✅ Cleaned data saved to database!")

✅ Cleaned data saved to database!
